In [14]:
%%writefile CSquareMatrixCUDA.cu
#include <iostream>
#include <vector>
#include <random>
#include <ctime>
#include <cstdlib>
#include <fstream>
#include <cuda_runtime.h>


class CSquareMatrix {
    private:
    size_t size_;
    std::vector<int> data_;
    
    public:
    CSquareMatrix(size_t size): size_(size), data_(size * size) {}


    int* getData() {
        return data_.data();
    }


    const int* getData() const {
        return data_.data();
    }


    int& operator() (size_t row, size_t col) {
        return data_[row * size_ + col];
    }


    const int& operator()(size_t row, size_t col) const {
        return data_[row * size_ + col];
    }


    size_t getSize() const {
        return size_;
    }


    void generateFullMatrix() {
        static std::random_device rd;
        static std::mt19937 gen(rd());
        std::uniform_int_distribution<> dis(1, 10);

        for (auto& num : data_) {
            num = dis(gen);
        }
    }
};


__global__ void multiplyMatricesCUDA(const int* A, const int* B, int* C, int N) {
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;
    
    if (row < N && col < N) {
        int sum = 0;
        for (int k = 0; k < N; k++) {
            sum += A[row * N + k] * B[k * N + col];
        }

        C[row * N + col] = sum;
    }
}


void writeOriginalMatricesFile(const CSquareMatrix& A, const CSquareMatrix& B) {
    if (A.getSize() != B.getSize()) {
            throw std::invalid_argument("Matrices must have the same size for multiplication");
    }

    std::ofstream file("original_matrices.txt");
    if (!file.is_open()) {
        throw std::runtime_error("Couldn't open the file");
    }

    size_t size = A.getSize();
    for (size_t i = 0; i < size; i++) {
        for (size_t j = 0; j < size; j++) {
            file << A(i, j) << " ";
        }

        file << "\n";
    }

    file << '\n';

    for (size_t i = 0; i < size; i++) {
        for (size_t j = 0; j < size; j++) {
            file << B(i, j) << " ";
        }
        file << "\n";
    }

    file.close();
}


void multiplicationCheckCUDA(const CSquareMatrix& A, const CSquareMatrix& B) {
    if (A.getSize() != B.getSize()) {
            throw std::invalid_argument("Matrices must have the same size for multiplication");
    }

    int N = A.getSize();
    const size_t bytes = N * N * sizeof(int);

    int *d_A, *d_B, *d_C;
    cudaMalloc(&d_A, bytes);
    cudaMalloc(&d_B, bytes);
    cudaMalloc(&d_C, bytes);

    cudaMemcpy(d_A, A.getData(), bytes, cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, B.getData(), bytes, cudaMemcpyHostToDevice);


    dim3 blockSize(8, 8);
    dim3 gridSize((N + blockSize.x- 1) / blockSize.x, (N + blockSize.y- 1) / blockSize.y);

    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    cudaEventRecord(start);
    multiplyMatricesCUDA<<<gridSize, blockSize>>>(d_A, d_B, d_C, N);
    cudaEventRecord(stop);

    cudaEventSynchronize(stop);

    float time_multiplication;
    cudaEventElapsedTime(&time_multiplication, start, stop);

    CSquareMatrix result(N);
    cudaMemcpy(result.getData(), d_C, bytes, cudaMemcpyDeviceToHost);

    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);
    cudaEventDestroy(start);
    cudaEventDestroy(stop);

    std::ofstream file("result_matrix.txt");
    if (!file.is_open()) {
        throw std::runtime_error("Couldn't open the file");
    }

    file << "Multiplication time: " << time_multiplication * 1000 << " microseconds\n";
    file << "Number of operations: " << (2 * N - 1) * N * N << "\n";

    for (size_t i = 0; i < N; i++) {
        for (size_t j = 0; j < N; j++) {
            file << result(i, j) << " ";
        }

        file << "\n";
    }

    file.close();
}


int main() {
    int N = 200;

    CSquareMatrix h_A(N), h_B(N);
    h_A.generateFullMatrix();
    h_B.generateFullMatrix();

    try {
        multiplicationCheckCUDA(h_A, h_B);
        //system("python verification_of_the_result.py");
        std::cout << "Matrices are multiplied\n";
    } catch (const std::exception& e) {
        std::cerr << "Error: " << e.what();
    }
}

Overwriting CSquareMatrixCUDA.cu
